# CMM vs Standard Momentum

This notebook compares the deep-learning weighted momentum signal (`CMM`) with traditional standard momentum (`SM`). It reports both raw factor results and size + industry neutralized factor results.

- `CMM`: trained model signal saved in `output/models/cmm/cmm_predictions.parquet`.
- `SM`: equal-weighted sum of daily log returns from `t-252` through `t-22`, skipping the most recent month as in the paper.
- Neutralization: month-by-month cross-sectional regression on `z_tmv` and `ind1` dummies; residuals are used as neutralized factors.
- Evaluation: monthly equal-weight decile portfolio charts plus equal-weight and value-weighted performance tables on the out-of-sample test split, with limit-up buy and limit-down sell constraints at rebalance.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

DATA_PATH = Path('../output/datasets/cmm_model_training_data.parquet')
PRED_PATH = Path('../output/models/cmm/cmm_predictions.parquet')
RETURN_COLS_PATH = Path('../output/datasets/cmm_return_window_columns.txt')
DAILY_DIR = Path('../../data/daily')
OUT_DIR = Path('../output/reports/model_compare')
OUT_DIR.mkdir(parents=True, exist_ok=True)

EVAL_SPLIT = 'test'  # final out-of-sample evaluation; validation folds can overlap.

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 160)
plt.style.use('seaborn-v0_8-whitegrid')


## 1. Load Signals and Build Standard Momentum

In [ ]:
return_cols = RETURN_COLS_PATH.read_text(encoding='utf-8').splitlines()

pred = pd.read_parquet(PRED_PATH)
pred['signal_date'] = pd.to_datetime(pred['signal_date'])

base_cols = ['stock_id', 'signal_date', 'trade_date', 'exit_date', 'target_1m_ret', 'ind1', 'z_tmv'] + return_cols
base = pd.read_parquet(DATA_PATH, columns=base_cols)
base['signal_date'] = pd.to_datetime(base['signal_date'])
base['trade_date'] = pd.to_datetime(base['trade_date'])
base['exit_date'] = pd.to_datetime(base['exit_date'])
base['ind1'] = base['ind1'].fillna('Unknown')

# Traditional momentum: equal-weighted past daily log returns over t-252:t-22.
# This matches the paper's standard momentum benchmark and skips the most recent month.
base['sm_signal'] = base[return_cols].sum(axis=1)

# Align CMM predictions with SM and controls.
df = pred[['stock_id', 'signal_date', 'split', 'cmm_signal_cs_z', 'target_1m_ret']].merge(
    base[['stock_id', 'signal_date', 'trade_date', 'exit_date', 'sm_signal', 'ind1', 'z_tmv']],
    on=['stock_id', 'signal_date'],
    how='inner',
)

if EVAL_SPLIT != 'test':
    raise ValueError("Use EVAL_SPLIT='test' for final comparison; validation predictions overlap across folds.")
df_eval = df[df['split'] == EVAL_SPLIT].copy()
if df_eval.duplicated(['stock_id', 'signal_date']).any():
    raise ValueError(f'Duplicate stock-month predictions detected in {EVAL_SPLIT} split.')

def normalize_daily_symbol(symbol: pd.Series) -> pd.Series:
    s = symbol.astype(str).str.strip()
    exchange = s.str[:2]
    code = s.str[2:]
    suffix = exchange.map({'SH': '.SH', 'SZ': '.SZ', 'BJ': '.BJ'}).fillna('')
    return code + suffix


def load_signal_market_caps(signal_dates: pd.Series) -> pd.DataFrame:
    pieces = []
    for d in sorted(pd.to_datetime(signal_dates.dropna().unique())):
        path = DAILY_DIR / f'{d:%Y-%m-%d}.csv'
        if not path.exists():
            continue
        daily = pd.read_csv(path, usecols=['date', 'symbol', 'tmv'])
        daily['signal_date'] = pd.to_datetime(daily['date'])
        daily['stock_id'] = normalize_daily_symbol(daily['symbol'])
        daily['market_cap'] = pd.to_numeric(daily['tmv'], errors='coerce')
        pieces.append(daily[['stock_id', 'signal_date', 'market_cap']])
    if not pieces:
        return pd.DataFrame(columns=['stock_id', 'signal_date', 'market_cap'])
    return pd.concat(pieces, ignore_index=True).drop_duplicates(['stock_id', 'signal_date'])


def limit_rate(stock_id: pd.Series, trade_date: pd.Series, st: pd.Series | None = None) -> pd.Series:
    code = stock_id.astype(str).str[:6]
    date = pd.to_datetime(trade_date)
    rate = pd.Series(0.10, index=stock_id.index, dtype=float)
    rate.loc[code.str.startswith('688') & (date >= pd.Timestamp('2019-07-22'))] = 0.20
    rate.loc[code.str.startswith('300') & (date >= pd.Timestamp('2020-08-24'))] = 0.20
    rate.loc[code.str.startswith(('8', '4'))] = 0.30
    if st is not None:
        rate.loc[pd.to_numeric(st, errors='coerce').fillna(0).astype(int).eq(1)] = 0.05
    return rate


def load_trade_flags(trade_dates: pd.Series) -> pd.DataFrame:
    pieces = []
    for d in sorted(pd.to_datetime(trade_dates.dropna().unique())):
        path = DAILY_DIR / f'{d:%Y-%m-%d}.csv'
        if not path.exists():
            continue
        daily = pd.read_csv(path, usecols=['date', 'symbol', 'close', 'preClose', 'ret', 'st'])
        daily['trade_date'] = pd.to_datetime(daily['date'])
        daily['stock_id'] = normalize_daily_symbol(daily['symbol'])
        rate = limit_rate(daily['stock_id'], daily['trade_date'], daily['st'])
        ret = pd.to_numeric(daily['ret'], errors='coerce')
        close = pd.to_numeric(daily['close'], errors='coerce')
        pre_close = pd.to_numeric(daily['preClose'], errors='coerce')
        daily['limit_up'] = (ret >= rate - 0.002) | (close >= pre_close * (1 + rate) * 0.998)
        daily['limit_down'] = (ret <= -rate + 0.002) | (close <= pre_close * (1 - rate) * 1.002)
        pieces.append(daily[['stock_id', 'trade_date', 'limit_up', 'limit_down']])
    if not pieces:
        return pd.DataFrame(columns=['stock_id', 'trade_date', 'limit_up', 'limit_down'])
    return pd.concat(pieces, ignore_index=True).drop_duplicates(['stock_id', 'trade_date'])


market_caps = load_signal_market_caps(df_eval['signal_date'])
df_eval = df_eval.merge(market_caps, on=['stock_id', 'signal_date'], how='left')

trade_flags = load_trade_flags(df_eval['trade_date'])
df_eval = df_eval.merge(trade_flags, on=['stock_id', 'trade_date'], how='left')
df_eval[['limit_up', 'limit_down']] = df_eval[['limit_up', 'limit_down']].fillna(False).astype(bool)

print(df_eval.shape)
print(df_eval['signal_date'].min(), df_eval['signal_date'].max())
print('industries:', df_eval['ind1'].nunique())
print('market cap coverage:', df_eval['market_cap'].notna().mean())
print('trade limit flags:', df_eval[['limit_up', 'limit_down']].mean().to_dict())
df_eval.head()


## 2. Size and Industry Neutralization

For each month, estimate:

```text
factor_i = a + b * z_tmv_i + industry_dummies_i + error_i
```

The residual `error_i` is the size + industry neutralized factor. `z_tmv` is the cross-sectionally standardized total market value feature produced during data cleaning.


In [ ]:
def neutralize_by_month(frame: pd.DataFrame, signal_col: str, size_col='z_tmv', industry_col='ind1') -> pd.Series:
    residuals = pd.Series(index=frame.index, dtype=float)
    for date, group in frame.groupby('signal_date', sort=True):
        idx = group.index
        y = pd.to_numeric(group[signal_col], errors='coerce').astype(float)
        size = pd.to_numeric(group[size_col], errors='coerce').astype(float).fillna(0.0)
        industry = group[industry_col].fillna('Unknown').astype(str)

        valid = y.notna()
        if valid.sum() < 50:
            residuals.loc[idx] = np.nan
            continue

        y_valid = y.loc[valid]
        x = pd.DataFrame({'const': 1.0, size_col: size.loc[valid]}, index=y_valid.index)
        dummies = pd.get_dummies(industry.loc[valid], prefix='ind', drop_first=True, dtype=float)
        x = pd.concat([x, dummies], axis=1)

        beta, *_ = np.linalg.lstsq(x.to_numpy(dtype=float), y_valid.to_numpy(dtype=float), rcond=None)
        fitted = x.to_numpy(dtype=float) @ beta
        residuals.loc[y_valid.index] = y_valid.to_numpy(dtype=float) - fitted
    return residuals


df_eval['cmm_neutral'] = neutralize_by_month(df_eval, 'cmm_signal_cs_z')
df_eval['sm_neutral'] = neutralize_by_month(df_eval, 'sm_signal')

neutral_check = df_eval[['cmm_signal_cs_z', 'cmm_neutral', 'sm_signal', 'sm_neutral', 'z_tmv']].corr()
neutral_check


## 3. Decile Portfolio Functions

The portfolio backtest rolls holdings month by month. At each rebalance date, stocks at the upper price limit cannot be bought, and stocks at the lower price limit cannot be sold.

In [ ]:
def assign_deciles(frame: pd.DataFrame, signal_col: str) -> pd.DataFrame:
    pieces = []
    for date, group in frame.groupby('signal_date', sort=True):
        group = group.dropna(subset=[signal_col, 'target_1m_ret']).copy()
        if len(group) < 100 or group[signal_col].nunique() < 10:
            continue
        group['decile'] = pd.qcut(group[signal_col].rank(method='first'), 10, labels=False) + 1
        pieces.append(group)
    return pd.concat(pieces, ignore_index=True)


def rebalance_one_decile(
    weights: dict[str, float],
    cash: float,
    target_weight: dict[str, float],
    can_buy: dict[str, bool],
    can_sell: dict[str, bool],
) -> tuple[dict[str, float], float]:
    target_names = set(target_weight)
    names = set(weights) | target_names
    new_weights = dict(weights)

    # Sell or reduce first. If the stock is limit-down, the sell order cannot be executed.
    for name in list(new_weights):
        current = new_weights.get(name, 0.0)
        target = target_weight.get(name, 0.0)
        if current > target:
            if can_sell.get(name, True):
                cash += current - target
                if target > 1e-12:
                    new_weights[name] = target
                else:
                    new_weights.pop(name, None)

    # Buy or increase positions with available cash. If the stock is limit-up, the buy order is skipped.
    buy_gaps = {}
    for name in target_names:
        current = new_weights.get(name, 0.0)
        target = target_weight.get(name, 0.0)
        if target > current and can_buy.get(name, True):
            buy_gaps[name] = target - current
    total_gap = sum(buy_gaps.values())
    if total_gap > 0 and cash > 0:
        scale = min(1.0, cash / total_gap)
        spent = 0.0
        for name, gap in buy_gaps.items():
            add = gap * scale
            new_weights[name] = new_weights.get(name, 0.0) + add
            spent += add
        cash -= spent

    return {k: v for k, v in new_weights.items() if v > 1e-12}, max(cash, 0.0)


def apply_month_return(weights: dict[str, float], cash: float, returns: dict[str, float]) -> tuple[float, dict[str, float], float]:
    stock_ret = {name: float(returns.get(name, 0.0)) for name in weights}
    portfolio_ret = sum(weight * stock_ret[name] for name, weight in weights.items())
    end_value = cash + sum(weight * (1.0 + stock_ret[name]) for name, weight in weights.items())
    if end_value <= 0:
        return portfolio_ret, {}, 0.0
    next_weights = {name: weight * (1.0 + stock_ret[name]) / end_value for name, weight in weights.items()}
    next_cash = cash / end_value
    return portfolio_ret, next_weights, next_cash


def target_weights_for_decile(month: pd.DataFrame, decile: int, weighting: str) -> dict[str, float]:
    target = month.loc[month['decile'] == decile, ['stock_id', 'market_cap']].copy()
    if target.empty:
        return {}
    if weighting == 'equal':
        return {name: 1.0 / len(target) for name in target['stock_id']}
    if weighting == 'value':
        cap = pd.to_numeric(target['market_cap'], errors='coerce').clip(lower=0).fillna(0.0)
        if cap.sum() <= 0:
            return {name: 1.0 / len(target) for name in target['stock_id']}
        weights = cap / cap.sum()
        return dict(zip(target['stock_id'], weights))
    raise ValueError("weighting must be 'equal' or 'value'")


def decile_returns(frame: pd.DataFrame, signal_col: str, weighting: str = 'equal') -> tuple[pd.DataFrame, pd.Series]:
    assigned = assign_deciles(frame, signal_col)
    dates = sorted(assigned['signal_date'].unique())
    portfolios = {decile: {'weights': {}, 'cash': 1.0} for decile in range(1, 11)}
    rows = []

    for date in dates:
        month = assigned[assigned['signal_date'] == date].copy()
        returns = month.set_index('stock_id')['target_1m_ret'].to_dict()
        can_buy = (~month.set_index('stock_id')['limit_up']).to_dict()
        can_sell = (~month.set_index('stock_id')['limit_down']).to_dict()
        row = {'signal_date': date}

        for decile in range(1, 11):
            target_weight = target_weights_for_decile(month, decile, weighting)
            state = portfolios[decile]
            weights, cash = rebalance_one_decile(state['weights'], state['cash'], target_weight, can_buy, can_sell)
            monthly_ret, next_weights, next_cash = apply_month_return(weights, cash, returns)
            portfolios[decile] = {'weights': next_weights, 'cash': next_cash}
            row[decile] = monthly_ret
        rows.append(row)

    monthly_decile = pd.DataFrame(rows).set_index('signal_date').sort_index()
    long_short = monthly_decile[10] - monthly_decile[1]
    return monthly_decile, long_short


def max_drawdown(monthly_returns: pd.Series) -> float:
    nav = (1 + monthly_returns.fillna(0)).cumprod()
    drawdown = nav / nav.cummax() - 1
    return float(drawdown.min())


def perf_stats(monthly_returns: pd.Series) -> dict:
    monthly_returns = monthly_returns.dropna()
    ann_ret = monthly_returns.mean() * 12
    ann_vol = monthly_returns.std(ddof=1) * np.sqrt(12)
    sharpe = ann_ret / ann_vol if ann_vol > 0 else np.nan
    t_stat = monthly_returns.mean() / (monthly_returns.std(ddof=1) / np.sqrt(len(monthly_returns))) if len(monthly_returns) > 1 else np.nan
    return {
        'months': len(monthly_returns),
        'annual_return': ann_ret,
        'annual_vol': ann_vol,
        'sharpe': sharpe,
        'max_drawdown': max_drawdown(monthly_returns),
        'monthly_win_rate': (monthly_returns > 0).mean(),
        'monthly_mean': monthly_returns.mean(),
        'monthly_t_stat': t_stat,
    }

signals = {
    'CMM': 'cmm_signal_cs_z',
    'Standard Momentum': 'sm_signal',
    'CMM Neutralized': 'cmm_neutral',
    'Standard Momentum Neutralized': 'sm_neutral',
}

decile_ret = {}
long_short = {}
decile_ret_vw = {}
long_short_vw = {}
for name, col in signals.items():
    decile_ret[name], long_short[name] = decile_returns(df_eval, col, weighting='equal')
    decile_ret_vw[name], long_short_vw[name] = decile_returns(df_eval, col, weighting='value')
    print(name, 'equal', decile_ret[name].shape, 'value', decile_ret_vw[name].shape)


## 4. Decile Line Charts

The y-axis is annualized average monthly return for each decile.

## 5. Decile Cumulative Return Curves

Each chart contains 10 lines, one for each decile portfolio after applying the same rebalance and price-limit constraints.

In [ ]:
def plot_decile_nav(name: str, filename: str):
    nav = (1 + decile_ret[name].fillna(0)).cumprod()
    fig, ax = plt.subplots(figsize=(10, 5.5))
    for decile in range(1, 11):
        ax.plot(nav.index, nav[decile], linewidth=1.4, label=f'D{decile}')
    ls_nav = (1 + long_short[name].fillna(0)).cumprod()
    ax.plot(ls_nav.index, ls_nav.values, color='black', linestyle='--', linewidth=2.4, label='Long-Short D10-D1')
    ax.set_title(f'{name} Decile Cumulative Net Value ({EVAL_SPLIT} split)')
    ax.set_xlabel('Signal date')
    ax.set_ylabel('Cumulative net value')
    ax.legend(ncol=5, fontsize=8, frameon=True)
    fig.tight_layout()
    path = OUT_DIR / filename
    fig.savefig(path, dpi=160, bbox_inches='tight')
    plt.close(fig)
    return path


decile_nav_paths = {
    'CMM': plot_decile_nav('CMM', f'decile_nav_cmm_{EVAL_SPLIT}.png'),
    'Standard Momentum': plot_decile_nav('Standard Momentum', f'decile_nav_standard_momentum_{EVAL_SPLIT}.png'),
    'CMM Neutralized': plot_decile_nav('CMM Neutralized', f'decile_nav_cmm_neutralized_{EVAL_SPLIT}.png'),
    'Standard Momentum Neutralized': plot_decile_nav('Standard Momentum Neutralized', f'decile_nav_standard_momentum_neutralized_{EVAL_SPLIT}.png'),
}
decile_nav_paths


## 6. Performance Metrics Table

Metrics are computed on the high-minus-low decile portfolio: long decile 10, short decile 1.

In [ ]:
perf = pd.DataFrame({name: perf_stats(ls) for name, ls in long_short.items()}).T
perf_vw = pd.DataFrame({name: perf_stats(ls) for name, ls in long_short_vw.items()}).T

perf_display = perf.copy()
for col in ['annual_return', 'annual_vol', 'max_drawdown', 'monthly_win_rate', 'monthly_mean']:
    perf_display[col] = perf_display[col].map(lambda x: f'{x:.2%}')
for col in ['sharpe', 'monthly_t_stat']:
    perf_display[col] = perf_display[col].map(lambda x: f'{x:.2f}')
perf_display['months'] = perf_display['months'].astype(int)

perf_vw_display = perf_vw.copy()
for col in ['annual_return', 'annual_vol', 'max_drawdown', 'monthly_win_rate', 'monthly_mean']:
    perf_vw_display[col] = perf_vw_display[col].map(lambda x: f'{x:.2%}')
for col in ['sharpe', 'monthly_t_stat']:
    perf_vw_display[col] = perf_vw_display[col].map(lambda x: f'{x:.2f}')
perf_vw_display['months'] = perf_vw_display['months'].astype(int)

perf.to_csv(OUT_DIR / f'performance_metrics_{EVAL_SPLIT}.csv', index_label='factor')
perf_vw.to_csv(OUT_DIR / f'performance_metrics_value_weighted_{EVAL_SPLIT}.csv', index_label='factor')
perf_display, perf_vw_display


## 7. Monthly Long-Short Curves